# GRU — Optuna Tuning

## Imports

In [ ]:
import optuna
import numpy as np
import torch
import torch.nn as nn
import os
import copy
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, r2_score

work_dir = os.getcwd()
print("work_dir:", work_dir)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device  :", DEVICE)


## Data Loader

In [ ]:
input_folder = os.path.join(work_dir, "data_normalized")

X_train = np.load(os.path.join(input_folder, "X_train.npy"))
y_train = np.load(os.path.join(input_folder, "y_train.npy"))
X_val = np.load(os.path.join(input_folder, "X_val.npy"))
y_val = np.load(os.path.join(input_folder, "y_val.npy"))
X_test = np.load(os.path.join(input_folder, "X_test.npy"))
y_test = np.load(os.path.join(input_folder, "y_test.npy"))

print(f"Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}")
print(f"y_train: [{y_train.min():.2f}, {y_train.max():.2f}]  std={y_train.std():.2f}")
assert X_train.shape[2] == 5, (
    f"Expected 5 input features [V, I, T, dV/dt, dI/dt], got {X_train.shape[2]}. "
    "Re-run 1_Preprocessing_Data.ipynb."
)
print("Input features confirmed: [voltage, current, temperature, dV/dt, dI/dt]")


def create_loader(X, y, batch_size=128, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(
        TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle, drop_last=False
    )


## Attention Module

In [ ]:
class Attention(nn.Module):
    """Additive (Bahdanau-style) attention over the time axis.
    Input : (B, T, H)   →   Output: (B, H)
    """

    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, x):  # x: (B, T, H)
        score = self.v(torch.tanh(self.attn(x)))  # (B, T, 1)
        weight = torch.softmax(score, dim=1)  # (B, T, 1)
        return (weight * x).sum(dim=1)  # (B, H)


## Objective

In [ ]:
def objective(trial):

    # Hyperparameters
    hidden_size = trial.suggest_int("hidden_size", 32, 192)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    huber_delta = trial.suggest_float("huber_delta", 0.5, 3.0)
    bidirectional = trial.suggest_categorical("bidirectional", [True, False])

    # Model
    H = hidden_size * (2 if bidirectional else 1)

    class _GRU(nn.Module):
        def __init__(self):
            super().__init__()
            self.gru = nn.GRU(
                input_size=5,  # always 5: V, I, T, dV/dt, dI/dt
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=dropout if num_layers > 1 else 0.0,
            )
            self.attn = Attention(H)  # must match GRU output size
            self.dropout = nn.Dropout(dropout)
            self.fc = nn.Linear(H, 1)

        def forward(self, x):
            out, _ = self.gru(x)  # (B, T, H)
            ctx = self.attn(out)  # (B, H)
            return self.fc(self.dropout(ctx)).squeeze(-1)

    # Data loaders
    _train = create_loader(X_train, y_train, batch_size)
    _val = create_loader(X_val, y_val, batch_size, shuffle=False)

    # Training setup
    m = _GRU().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-5)
    crit = nn.HuberLoss(delta=huber_delta)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=40, eta_min=1e-6)

    best_composite = float("inf")
    best_state = copy.deepcopy(m.state_dict())

    for epoch in range(40):
        # Train
        m.train()
        for xb, yb in _train:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            crit(m(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        sch.step()

        # Validate
        m.eval()
        preds_v, trues_v = [], []
        with torch.no_grad():
            for xb, yb in _val:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds_v.append(m(xb).cpu().numpy())
                trues_v.append(yb.cpu().numpy())

        y_pred_v = np.concatenate(preds_v)
        y_true_v = np.concatenate(trues_v)

        val_rmse = np.sqrt(mean_squared_error(y_true_v, y_pred_v))
        val_r2 = r2_score(y_true_v, y_pred_v)

        ALPHA = 0.2
        r2_term = 1.0 - max(val_r2, -1.0)  # in [0, 2], lower is better
        composite = ALPHA * val_rmse + (1.0 - ALPHA) * r2_term

        # Report composite (not raw loss) so the pruner compares trials fairly
        trial.report(composite, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if composite < best_composite:
            best_composite = composite
            best_state = copy.deepcopy(m.state_dict())

    return best_composite


## Study

In [ ]:
study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)
study.optimize(objective, n_trials=50)


## Results

In [ ]:
print("=" * 55)
print("Best composite :", study.best_value)
print("Best params    :", study.best_params)
print("=" * 55)

# Optuna visualizations
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.tight_layout()
plt.show()

optuna.visualization.matplotlib.plot_param_importances(study)
plt.tight_layout()
plt.show()

optuna.visualization.matplotlib.plot_slice(study)
plt.tight_layout()
plt.show()


## Retrain Best Config & Evaluate on Test

In [ ]:
# Retrain best config on train+val and evaluate on test

bp = study.best_params
print("Retraining with best params:", bp)

H_best = bp["hidden_size"] * (2 if bp["bidirectional"] else 1)


class GRUBest(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(
            input_size=5,
            hidden_size=bp["hidden_size"],
            num_layers=bp["num_layers"],
            batch_first=True,
            bidirectional=bp["bidirectional"],
            dropout=bp["dropout"] if bp["num_layers"] > 1 else 0.0,
        )
        self.attn = Attention(H_best)
        self.dropout = nn.Dropout(bp["dropout"])
        self.fc = nn.Linear(H_best, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(self.dropout(self.attn(out))).squeeze(-1)


model = GRUBest().to(DEVICE)
total_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters : {total_p:,}")
print(f"Bidirectional: {bp['bidirectional']}   H={H_best}")

# Combine train + val for final training
X_fit = np.concatenate([X_train, X_val])
y_fit = np.concatenate([y_train, y_val])
fit_loader = create_loader(X_fit, y_fit, bp["batch_size"])
test_loader = create_loader(X_test, y_test, bp["batch_size"], shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=bp["lr"], weight_decay=1e-5)
criterion = nn.HuberLoss(delta=bp["huber_delta"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=100, eta_min=1e-6
)

EPOCHS = 100
PATIENCE = 20
best_val_loss = float("inf")
best_state = copy.deepcopy(model.state_dict())
counter = 0

split_idx = int(len(X_fit) * 0.85)
fit_tr_loader = create_loader(X_fit[:split_idx], y_fit[:split_idx], bp["batch_size"])
fit_vl_loader = create_loader(
    X_fit[split_idx:], y_fit[split_idx:], bp["batch_size"], shuffle=False
)

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    bl = []
    for xb, yb in fit_tr_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bl.append(criterion(model(xb.detach()), yb.detach()).item())
    train_loss = np.mean(bl)

    model.eval()
    vl = []
    with torch.no_grad():
        for xb, yb in fit_vl_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            vl.append(criterion(model(xb), yb).item())
    val_loss = np.mean(vl)

    scheduler.step()
    train_losses.append(np.sqrt(train_loss))
    val_losses.append(np.sqrt(val_loss))

    print(
        f"Ep {epoch:03d}/{EPOCHS} | train={np.sqrt(train_loss):.4f} | val={np.sqrt(val_loss):.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1
    if counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)

# Training curve
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Train RMSE")
plt.plot(val_losses, label="Val RMSE")
plt.legend()
plt.title("Final GRU Training Curve")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.grid()
plt.tight_layout()
plt.show()


## Test Metrics

In [ ]:
# Test evaluation
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
y_pred = np.concatenate(preds)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

print("=" * 45)
print(f"Test RMSE : {test_rmse:.4f} %")
print(f"Test R²   : {test_r2:.6f}")
print(f"Pred std  : {y_pred.std():.4f}  (test std: {y_test.std():.4f})")
print("=" * 45)

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred, alpha=0.3, s=8)
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
axes[0].plot([lo, hi], [lo, hi], "r--", lw=1)
axes[0].set_xlabel("True SoH")
axes[0].set_ylabel("Predicted SoH")
axes[0].set_title(f"Prediction vs Ground Truth\nRMSE={test_rmse:.3f}  R²={test_r2:.4f}")
axes[0].grid(alpha=0.3)

residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=8)
axes[1].axhline(0, color="r", lw=1)
axes[1].set_xlabel("Predicted SoH")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual Plot")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Save Best Params & Weights

In [ ]:
# Save best params
import json

best_params_out = {
    "hidden_size": bp["hidden_size"],
    "num_layers": bp["num_layers"],
    "dropout": round(bp["dropout"], 6),
    "lr": round(bp["lr"], 8),
    "batch_size": bp["batch_size"],
    "huber_delta": round(bp["huber_delta"], 4),
    "bidirectional": bp["bidirectional"],
    "test_rmse": round(float(test_rmse), 4),
    "test_r2": round(float(test_r2), 6),
}

log_dir = os.path.join(work_dir, "log")
os.makedirs(log_dir, exist_ok=True)
out_path = os.path.join(log_dir, "gru_best_params.json")
with open(out_path, "w") as f:
    json.dump(best_params_out, f, indent=4)
print(f"Best params saved → {out_path}")
print(json.dumps(best_params_out, indent=4))

model_dir = os.path.join(work_dir, "model")
os.makedirs(model_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(model_dir, "gru_optuna_best.pt"))
print(f"Model weights saved → {model_dir}/gru_optuna_best.pt")
